### Using XGBoost with Optuna


In [1]:
import numpy as np
import pandas as pd

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

import xgboost as xgb
from sklearn.datasets import load_iris


import warnings
warnings.filterwarnings('ignore')

In [4]:

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Creating DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Defining a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-03-10 08:15:03,988] A new study created in memory with name: no-name-be61224d-4ee3-4a90-92b1-bbebf3592f1d
[I 2026-03-10 08:15:10,410] Trial 0 finished with value: 1.0 and parameters: {'lambda': 3.516375710193424e-05, 'alpha': 2.57431813921617e-06, 'eta': 0.17480603358055685, 'gamma': 0.0616429731012901, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.4277023782898663, 'colsample_bytree': 0.6013398044501397}. Best is trial 0 with value: 1.0.
[I 2026-03-10 08:15:15,265] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.035877137469012235, 'alpha': 2.6174916006867184e-06, 'eta': 0.06444971295389378, 'gamma': 0.010839247328073358, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.6642735130193158, 'colsample_bytree': 0.8039921359809834}. Best is trial 0 with value: 1.0.
[I 2026-03-10 08:15:15,406] Trial 2 pruned. Trial was pruned at iteration 1.
[I 2026-03-10 08:15:15,525] Trial 3 pruned. Trial was pruned at iteration 4.
[I 2026-03-10 08:15:15,584] Trial 4 pru

Best trial: {'lambda': 3.516375710193424e-05, 'alpha': 2.57431813921617e-06, 'eta': 0.17480603358055685, 'gamma': 0.0616429731012901, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.4277023782898663, 'colsample_bytree': 0.6013398044501397}
Best accuracy: 1.0


In [5]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()